# 001 · Numbers

> **Syntax › Data Types** · Python 3.14

Python has far fewer numeric types than C#: one integer type (`int`) with **unlimited precision**, one binary floating-point type (`float`), plus `complex` built in and `Decimal`/`Fraction` in the standard library.

**In this notebook you will learn:**

- why Python's `int` never overflows (and what replaces `int`/`long`/`BigInteger`)
- how `float` behaves (IEEE 754 double) and how to compare floats safely
- when to use `decimal.Decimal` (C#'s `decimal`) and `fractions.Fraction`
- numeric literals: underscores, hex/binary/octal, scientific notation
- rounding (banker's rounding!), conversions, the `math` module, `inf` and `nan`

## C# mental model

| C# | Python |
|---|---|
| `byte`, `short`, `int`, `long`, `BigInteger` (+ unsigned variants) | `int` — one type, arbitrary precision |
| `double` | `float` (always 64-bit IEEE 754) |
| `float` (32-bit) | no built-in equivalent |
| `decimal` | `decimal.Decimal` (standard library) |
| `System.Numerics.Complex` | `complex` (built in, literal `3+4j`) |
| — | `fractions.Fraction` (exact rationals) |

⚠️ **Gotcha:** the word `float` means a **64-bit double** in Python, not C#'s 32-bit `float`.

## `int` — arbitrary precision, no overflow

In C#, `int` is 32 bits and silently wraps around on overflow (unless you use `checked`). You switch to `long` or `BigInteger` when you need more range.

```csharp
// C#
int big = int.MaxValue;
big++;                                  // wraps to -2147483648 (unchecked by default)
checked { big++; }                      // OverflowException
var huge = BigInteger.Pow(2, 100);      // need System.Numerics for this
```

Python's `int` grows as needed — it is effectively `BigInteger` everywhere, so there is no overflow and no `checked` keyword. The only practical limit is memory. (As a safety measure, converting an int with more than 4300 digits to/from `str` raises `ValueError` by default — see `sys.set_int_max_str_digits`.)

In [ ]:
big = 2_147_483_647      # C#'s int.MaxValue
print(big + 1)           # just keeps going: 2147483648
print(2 ** 100)          # exact, no BigInteger needed
print(type(2 ** 100))    # still plain int

In [ ]:
import math

print(math.factorial(30))         # 265252859812191058636308480000000
print(len(str(2 ** 10_000)))      # a 3011-digit number, computed instantly

## Integer division and modulo

`/` **always** returns a `float` — even for two ints. Use `//` for floor division. Note that `//` and `%` round towards **negative infinity**, whereas C#'s `/` and `%` truncate towards zero. *(More in Operators › 001 · Arithmetic and Assignment Operators.)*

```csharp
// C#
Console.WriteLine(7 / 2);    // 3   (int division)
Console.WriteLine(-7 / 2);   // -3  (truncates toward zero)
Console.WriteLine(-7 % 2);   // -1
```

In [ ]:
print(7 / 2)       # 3.5  — true division, always float
print(7 // 2)      # 3    — floor division
print(-7 // 2)     # -4   — floors, C# would give -3
print(-7 % 2)      # 1    — sign follows the divisor, C# would give -1
print(divmod(17, 5))  # (3, 2) — quotient and remainder at once

## Numeric literals

Underscores can be used as digit separators (like C#'s `1_000_000`, added in C# 7). Prefixes for hex, binary, and octal work like C#, with octal using `0o`.

| Literal | C# | Python |
|---|---|---|
| Digit separators | `1_000_000` | `1_000_000` |
| Hex | `0xFF` | `0xFF` |
| Binary | `0b1010` | `0b1010` |
| Octal | — | `0o17` |
| Scientific | `1.5e3` | `1.5e3` |
| Type suffixes | `10L`, `1.5f`, `2.5m` | none — no suffixes exist |

In [ ]:
population = 8_100_000_000
print(population)

print(0xFF, 0b1010, 0o17)     # 255 10 15
print(1.5e3, 2e-3)            # 1500.0 0.002

# And back to strings:
print(hex(255), bin(10), oct(15))   # '0xff' '0b1010' '0o17'

`int()` can parse other bases too — base `0` means "infer from the prefix":

In [ ]:
print(int("ff", 16))       # 255  (C#: Convert.ToInt32("ff", 16))
print(int("1010", 2))      # 10
print(int("0x1F", 0))      # 31   — base inferred from prefix
print(int("1_000"))        # 1000 — underscores are accepted in strings too

## `float` — IEEE 754 double

`float` is exactly C#'s `double`, so it has exactly the same rounding behaviour. The famous example:

```csharp
// C#
Console.WriteLine(0.1 + 0.2 == 0.3);   // False
```

In [ ]:
print(0.1 + 0.2)           # 0.30000000000000004
print(0.1 + 0.2 == 0.3)    # False — same as C#

Compare floats with a tolerance using `math.isclose` (defaults to a relative tolerance of `1e-09`). There is no direct built-in in .NET — you normally write `Math.Abs(a - b) < epsilon` yourself.

In [ ]:
import math

print(math.isclose(0.1 + 0.2, 0.3))                  # True
print(math.isclose(1000.0, 1000.1, rel_tol=1e-3))    # True — within 0.1 %
print(math.isclose(0.0, 1e-12, abs_tol=1e-9))        # near zero you need abs_tol

Useful float limits and introspection:

In [ ]:
import sys

print(sys.float_info.max)          # 1.7976931348623157e+308  (C#: double.MaxValue)
print(sys.float_info.epsilon)      # 2.220446049250313e-16
print((2.5).is_integer())          # False
print((0.1).as_integer_ratio())    # the exact fraction actually stored for 0.1

## Infinity and NaN

Python supports IEEE infinities and NaN just like `double.PositiveInfinity` and `double.NaN`, but there are no named constants on `float` — create them with `float("inf")` / `float("nan")` or use `math.inf` / `math.nan`.

⚠️ **Gotcha:** unlike C#, dividing a float by zero does **not** give infinity — it raises `ZeroDivisionError`.

In [ ]:
import math

print(math.inf, -math.inf, float("inf") > 10**300)   # inf -inf True
print(1e308 * 10)                                    # overflow -> inf

nan = float("nan")
print(nan == nan)          # False — NaN is never equal to anything (same as C#)
print(math.isnan(nan))     # True  (C#: double.IsNaN)

try:
    1.0 / 0.0              # C#: double.PositiveInfinity
except ZeroDivisionError as e:
    print(f"{type(e).__name__}: {e}")

## `decimal.Decimal` — for money

C#'s `decimal` is a built-in 128-bit decimal type with 28–29 significant digits. Python's equivalent is `decimal.Decimal` in the standard library: base-10, exact for decimal fractions, with a configurable precision (28 significant digits by default).

```csharp
// C#
decimal price = 0.1m + 0.2m;           // 0.3 exactly
```

⚠️ **Gotcha:** construct `Decimal` from a **string**, not a float — a float has already lost precision before `Decimal` ever sees it.

In [ ]:
from decimal import Decimal

print(Decimal("0.1") + Decimal("0.2"))                     # 0.3 exactly
print(Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))   # True
print(Decimal(0.1))   # from float: 0.1000000000000000055511151231257827...

Rounding money to 2 places uses `quantize` with an explicit rounding mode — like `Math.Round(x, 2, MidpointRounding.AwayFromZero)`:

In [ ]:
from decimal import Decimal, ROUND_HALF_UP, ROUND_HALF_EVEN

amount = Decimal("2.675")
print(amount.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))     # 2.68
print(amount.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN))   # 2.68 (7 is odd -> round to even 8)
print(Decimal("2.665").quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN))  # 2.66

Mixing `Decimal` and `float` is not allowed (C# also requires an explicit cast between `decimal` and `double`); mixing with `int` is fine:

In [ ]:
print(Decimal("1.5") * 2)          # 3.0 — int is fine

try:
    Decimal("1.5") + 0.5
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## `fractions.Fraction` — exact rationals

There is no C# equivalent in the BCL. `Fraction` stores an exact numerator/denominator pair, which is handy for maths where any rounding is unacceptable.

In [ ]:
from fractions import Fraction

third = Fraction(1, 3)
print(third + third + third)        # 1
print(Fraction(3, 4) * 2)           # 3/2
print(Fraction("0.25"))             # 1/4
print(float(Fraction(1, 3)))        # 0.3333333333333333

## `complex` — built in

Complex numbers are a built-in type with `j` as the imaginary suffix (engineering notation). C# has `System.Numerics.Complex`.

In [ ]:
z = 3 + 4j
print(z, type(z))
print(z.real, z.imag)      # 3.0 4.0
print(abs(z))              # 5.0 — magnitude (C#: Complex.Abs / z.Magnitude)
print(z * 1j)              # (-4+3j)

## Rounding — banker's rounding by default

`round(x)` rounds halves to the nearest **even** number ("banker's rounding"). This matches C#'s `Math.Round`, whose default is `MidpointRounding.ToEven` — but surprises many people coming from other languages.

```csharp
// C#
Console.WriteLine(Math.Round(2.5));   // 2
Console.WriteLine(Math.Round(3.5));   // 4
Console.WriteLine(Math.Round(2.5, MidpointRounding.AwayFromZero));   // 3
```

In [ ]:
print(round(0.5), round(1.5), round(2.5), round(3.5))   # 0 2 2 4
print(round(-2.5))                                      # -2

print(type(round(2.5)))       # int   — with no ndigits, round returns an int
print(round(3.14159, 2))      # 3.14  — with ndigits, it returns a float

⚠️ **Gotcha:** `round(2.675, 2)` gives `2.67`, not `2.68` — not because of banker's rounding, but because `2.675` is actually stored as `2.67499999…` in binary. Use `Decimal` when exact decimal rounding matters.

In [ ]:
print(round(2.675, 2))        # 2.67
print(f"{2.675:.20f}")        # 2.67499999999999982236 — slightly below 2.675

## Converting between `int` and `float`

| C# | Python |
|---|---|
| `(int)3.9` → `3` (truncates) | `int(3.9)` → `3` (truncates toward zero) |
| `Math.Floor(x)` | `math.floor(x)` → returns an `int` |
| `Math.Ceiling(x)` | `math.ceil(x)` → returns an `int` |
| `Math.Truncate(x)` | `math.trunc(x)` or `int(x)` |
| `(double)n` | `float(n)` |

Mixed arithmetic promotes automatically: `int` + `float` → `float`. *(See Data Types › 006 · Type Conversion and Checking.)*

In [ ]:
import math

print(int(3.9), int(-3.9))              # 3 -3   — truncation
print(math.floor(-2.5), math.ceil(2.1)) # -3 3   — ints, not floats (C# returns double)
print(float(7), 1 + 2.0)                # 7.0 3.0

⚠️ **Gotcha:** very large ints lose precision when converted to `float`, and ints too big for a double raise `OverflowError`:

In [ ]:
print(float(2**53 + 1) == float(2**53))   # True — precision lost beyond 2**53

try:
    float(10 ** 400)
except OverflowError as e:
    print(f"{type(e).__name__}: {e}")

## The `math` module and built-ins

The equivalent of `System.Math` is the `math` module. A few things are built-in functions instead: `abs`, `round`, `min`, `max`, `sum`, `pow`, `divmod`.

In [ ]:
import math

print(abs(-5), min(3, 1, 2), max(3, 1, 2), sum([1, 2, 3]))   # built-ins
print(2 ** 10, pow(2, 10), pow(2, 10, 1000))   # ** is the power operator; 3-arg pow is modular
print(math.sqrt(16), math.pi, math.e)
print(math.gcd(12, 18), math.lcm(4, 6))      # 6 12
print(math.log(math.e), math.log10(1000), math.log2(8))
print(math.hypot(3, 4))                      # 5.0

## Checking numeric types

`bool` is a subclass of `int` (*see Data Types › 004 · Booleans and Truthiness*). To accept "any number", use the abstract base classes in `numbers`. Note that `Decimal` deliberately does **not** register as `numbers.Real` (it refuses to mix with `float`), only as `numbers.Number`:

In [ ]:
import numbers
from decimal import Decimal

for value in [42, 3.14, 2 + 1j, Decimal("1.5")]:
    print(f"{value!s:>6}: {type(value).__name__:<8} Real? {isinstance(value, numbers.Real)}")

## Summary

| Task | C# | Python |
|---|---|---|
| Integer (any size) | `int` / `long` / `BigInteger` | `int` |
| Overflow check | `checked { }` | not needed — no overflow |
| Double precision | `double` | `float` |
| Money | `decimal`, `2.5m` | `Decimal("2.5")` |
| Complex | `new Complex(3, 4)` | `3 + 4j` |
| Digit separators | `1_000_000` | `1_000_000` |
| Integer division | `7 / 2` (ints) | `7 // 2` |
| True division | `7.0 / 2` | `7 / 2` |
| Power | `Math.Pow(2, 10)` | `2 ** 10` |
| Round half to even | `Math.Round(x)` | `round(x)` |
| Float compare | `Math.Abs(a - b) < eps` | `math.isclose(a, b)` |
| NaN check | `double.IsNaN(x)` | `math.isnan(x)` |
| Parse hex | `Convert.ToInt32(s, 16)` | `int(s, 16)` |

## Exercises

1. Compute `2 ** 64` and `2 ** 64 - 1` (C#'s `ulong.MaxValue`). Then compute the number of digits in `100!` (factorial).
2. Add `0.1` ten times in a loop using `float`, then using `Decimal`. Compare each result to `1`.
3. Write `round_half_up(x: float) -> int` that rounds `2.5 → 3` and `-2.5 → -3` (away from zero), using `Decimal`.
4. Without running it, predict `-9 // 4`, `-9 % 4`, and `int(-9 / 4)`. Then check.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
import math
print(2 ** 64, 2 ** 64 - 1)
print(len(str(math.factorial(100))))   # 158

# 2
from decimal import Decimal
f = 0.0
d = Decimal("0")
for _ in range(10):
    f += 0.1
    d += Decimal("0.1")
print(f, f == 1)   # 0.9999999999999999 False
print(d, d == 1)   # 1.0 True

# 3
from decimal import Decimal, ROUND_HALF_UP
def round_half_up(x: float) -> int:
    # str(x) avoids the binary representation error; ROUND_HALF_UP rounds away from zero
    return int(Decimal(str(x)).quantize(Decimal("1"), rounding=ROUND_HALF_UP))
print(round_half_up(2.5), round_half_up(-2.5))   # 3 -3

# 4 — floor division rounds toward negative infinity
print(-9 // 4)      # -3
print(-9 % 4)       # 3   (because -3 * 4 + 3 == -9)
print(int(-9 / 4))  # -2  (truncates -2.25 toward zero, like C#'s -9 / 4)
```

</details>